In [6]:
import csv
import re
import os

VECTOR_EXT_PATTERN = re.compile(r'^.+\.(txt|dat|vec)$', re.IGNORECASE)
RESULTS_EXT_PATTERN = re.compile(r'^.+\.(csv|txt)$', re.IGNORECASE)

NUMERIC_RE = re.compile(r'^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$')

CSV_HEADER = ["Operation", "Size", "Time"]
def _validate_filename(f, pattern, kind):
    if not isinstance(f, str) or not f.strip():
        raise ValueError(f"Имя файла для {kind} пустое или не строка")
    if not pattern.match(f):
        raise ValueError(
            f"Ошибочный формат имени файла для {kind}: '{f}' "
            f"(ожидается расширение из допустимых)"
        )

def _check_readable(f):
    if not os.path.exists(f):
        raise FileNotFoundError(f"Файл не существует: {f}")
    if not os.path.isfile(f):
        raise IsADirectoryError(f"Путь не является файлом: {f}")
    if not os.access(f, os.R_OK):
        raise PermissionError(f"Нет прав на чтение: {f}")


def _check_writable(f):
    parent = os.path.dirname(os.path.abspath(f)) or "."
    if not os.path.isdir(parent):
        raise FileNotFoundError(f"Каталог для записи не существует: {parent}")
    if os.path.isdir(f):
        raise IsADirectoryError(f"Имя занято каталогом: {f}")
    if not os.access(parent, os.W_OK):
        raise PermissionError(f"Нет прав на запись в каталог: {parent}")


def write_vector(f, v):
    _validate_filename(f, VECTOR_EXT_PATTERN, "вектора")
    _check_writable(f)

    if v is None:
        raise ValueError("Вектор не задан (None)")

    values = []
    for i, x in enumerate(v):
        try:
            values.append(float(x))
        except (TypeError, ValueError) as e:
            raise ValueError(f"Недопустимое значение в векторе, индекс {i}: {x!r}") from e

    try:
        with open(f, "w", encoding="utf-8") as a:
            a.write(" ".join(repr(x) if isinstance(x, float) else str(x) for x in values))
    except OSError as e:
        raise IOError(f"Ошибка записи файла '{f}': {e}") from e


def read_vector(f):
    _validate_filename(f, VECTOR_EXT_PATTERN, "вектора")
    _check_readable(f)

    try:
        with open(f, "r", encoding="utf-8") as a:
            content = a.read()
    except OSError as e:
        raise IOError(f"Ошибка чтения файла '{f}': {e}") from e
    except UnicodeDecodeError as e:
        raise ValueError(f"Ошибочный формат файла '{f}': не UTF-8") from e

    tokens = content.split()
    if not tokens:
        raise ValueError(f"Файл '{f}' пуст — вектор не может быть прочитан")

    v = []
    for i, t in enumerate(tokens):
        if not NUMERIC_RE.match(t):
            raise ValueError(
                f"Ошибочный формат файла '{f}': токен #{i} = {t!r} не является числом"
            )
        v.append(float(t))

    if len(v) == 0:
        raise ValueError(f"Несоответствие размеров: вектор из '{f}' пуст")
    return v


def write_results(f, r):
    _validate_filename(f, RESULTS_EXT_PATTERN, "результатов")
    _check_writable(f)

    if r is None:
        raise ValueError("Результаты не заданы (None)")

    normalized = []
    for i, row in enumerate(r):
        if not isinstance(row, (list, tuple)):
            raise ValueError(f"Строка #{i} результата не является списком/кортежем: {row!r}")
        if len(row) != 3:
            raise ValueError(
                f"Несоответствие размеров: строка #{i} имеет длину {len(row)}, ожидается 3"
            )
        op, size, t = row
        if not isinstance(op, str) or not op:
            raise ValueError(f"Недопустимое значение Operation в строке #{i}: {op!r}")
        if not isinstance(size, int) or size <= 0:
            raise ValueError(f"Недопустимое значение Size в строке #{i}: {size!r}")
        try:
            t = float(t)
        except (TypeError, ValueError) as e:
            raise ValueError(f"Недопустимое значение Time в строке #{i}: {t!r}") from e
        if t < 0:
            raise ValueError(f"Time не может быть отрицательным (строка #{i}): {t}")
        normalized.append([op, size, t])

    try:
        with open(f, "w", newline="", encoding="utf-8") as a:
            w = csv.writer(a)
            w.writerow(CSV_HEADER)
            for x in normalized:
                w.writerow(x)
    except OSError as e:
        raise IOError(f"Ошибка записи файла '{f}': {e}") from e

In [7]:
write_vector("v.txt", [1, 2.5, -3])
print(read_vector("v.txt"))

write_results("r.csv", [["mm", 10, 0.001]])
print(open("r.csv").read())

for name in ["v", "v.json", "v.txt.exe", "", None, ".txt"]:
    try:
        write_vector(name, [1])
        print("FAIL", name, "прошло")
    except ValueError as e:
        print("PASS", name, "->", e)

open("bad.txt", "w").write("1 2 abc 4")
try:
    read_vector("bad.txt")
except ValueError as e:
    print("PASS bad.txt ->", e)

try:
    write_results("r2.csv", [["x", 10]])
except ValueError as e:
    print("PASS r2 ->", e)

[1.0, 2.5, -3.0]
Operation,Size,Time
mm,10,0.001

PASS v -> Ошибочный формат имени файла для вектора: 'v' (ожидается расширение из допустимых)
PASS v.json -> Ошибочный формат имени файла для вектора: 'v.json' (ожидается расширение из допустимых)
PASS v.txt.exe -> Ошибочный формат имени файла для вектора: 'v.txt.exe' (ожидается расширение из допустимых)
PASS  -> Имя файла для вектора пустое или не строка
PASS None -> Имя файла для вектора пустое или не строка
PASS .txt -> Ошибочный формат имени файла для вектора: '.txt' (ожидается расширение из допустимых)
PASS bad.txt -> Ошибочный формат файла 'bad.txt': токен #2 = 'abc' не является числом
PASS r2 -> Несоответствие размеров: строка #0 имеет длину 2, ожидается 3
